# Feeling of Success Split Analysis

This notebook reads the extracted sample manifest, builds one selected split policy, assigns train/eval/test splits by whole split groups, and optionally writes final manifest-schema CSVs for model training.

In [1]:
from pathlib import Path

# Input manifest generated by scripts/convert_h5_to_png_samples.py.
# for complete training
MANIFEST_CSV = Path("../feeling-of-success/manifest.csv")
OUTPUT_DIR = Path("../feeling-of-success")

# for testing
# MANIFEST_CSV = Path("../dataset-limited/manifest.csv")
# OUTPUT_DIR = Path("../dataset-limited")


In [2]:
# Tunable notebook settings.
SEED = 40
SPLIT_RATIOS = {"train": 0.70, "eval": 0.15, "test": 0.15}
SPLIT_GROUP_STRATEGY = "object_type"  # choose "object_type" or "object_type_dataset"

# Set these True when you want this notebook to write CSV files.
WRITE_SPLIT_CSVS = True
WRITE_VISUALLY_CHALLENGING_SPLIT_CSVS = True

# Object types likely to be challenging for a vision-only model.
VISUALLY_CHALLENGING_OBJECT_TYPES = [
    "3d_printed_black_cylinder_gear",
    "3d_printed_white_ball",
    "black_metallic_candle_cage",
    "black_plastic_half_cylinder",
    "blue_bottle_fuel_treatment",
    "blue_painted_glass",
    "candle_in_glass",
    "dark_blue_sphere",
    "edge_shave_gel",
    "glass_candle_holder",
    "green_and_black_sphere",
    "green_plastic_cup",
    "isopropyl_alcohol",
    "metal_can",
    "metal_cylinder_with_holes",
    "monofilament_line",
    "ogx_shampoo",
    "peppermint_altoids_box",
    "pink_blue_coke_bottle",
    "pink_glass_glass",
    "pino_silvestre",
    "plastic_watering_can",
    "ponds_dry_skin_cream",
    "purple_small_plastic_fruit",
    "red_bull",
    "soda_can",
    "soft_blue_cylinder",
    "soft_blue_hexagon",
    "soft_red_cube",
    "spam",
    "stuffed_beachball",
    "tomato_paste_in_metal_can",
    "translucent_turquoise_cup",
    "tuna_can",
    "wiry_sphere",
]


In [3]:
import csv
import html
import random
import re
from collections import defaultdict
from pathlib import Path
from IPython.display import HTML, display

FINAL_MANIFEST_FIELDNAMES = [
    "episode_id",
    "dataset_name",
    "task_id",
    "task_name",
    "object_id",
    "object_class",
    "condition_class",
    "clip_type",
    "result",
    "failure_mode",
    "rgb_path",
    "tactile_path",
    "force_path",
    "metadata_path",
    "split_group",
]


def parse_bool(value):
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes", "success"}:
        return True
    if normalized in {"false", "0", "no", "failure"}:
        return False
    raise ValueError(f"Could not parse boolean value: {value!r}")


def h5_number(source_h5):
    match = re.search(r"_(\d+)$", Path(source_h5).stem)
    if not match:
        raise ValueError(f"Could not find trailing _<number> in source_h5: {source_h5}")
    return match.group(1)


def split_group_for(row):
    if SPLIT_GROUP_STRATEGY == "object_type":
        return row["object_type"]
    if SPLIT_GROUP_STRATEGY == "object_type_dataset":
        return f"{row['object_type']}_{row['episode_id']}"
    raise ValueError(f"Unknown split group strategy: {SPLIT_GROUP_STRATEGY}")


def final_manifest_row(row, split_group, visually_challenging_objects):
    result = row["is_gripping"]
    return {
        "episode_id": row["episode_id"],
        "dataset_name": "feeling_of_success",
        "task_id": row["source_index"],
        "task_name": "grasp_outcome",
        "object_id": row["object_type"],
        "object_class": "visually_challenging" if row["object_type"] in visually_challenging_objects else "normal",
        "condition_class": "",
        "clip_type": "before-during-after-images",
        "result": result,
        "failure_mode": "" if result else "unknown",
        "rgb_path": row["rgb_path"],
        "tactile_path": row["tactile_path"],
        "force_path": "",
        "metadata_path": row["metadata_path"],
        "split_group": split_group,
    }


def display_table(rows, title=None, limit=20):
    rows = list(rows)
    if title:
        display(HTML(f"<h3>{html.escape(title)}</h3>"))
    if not rows:
        display(HTML("<em>No rows</em>"))
        return
    columns = list(rows[0].keys())
    if limit:
        shown = rows[:limit]
    else:
        shown = rows
    header = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body = []
    for row in shown:
        body.append("<tr>" + "".join(f"<td>{html.escape(str(row.get(column, '')))}</td>" for column in columns) + "</tr>")
    note = "" if limit is not None and len(rows) <= limit else f"<p><em>Showing {limit} of {len(rows)} rows.</em></p>"
    display(HTML(f"""
    <style>
      table.split-table {{ border-collapse: collapse; font-size: 13px; }}
      table.split-table th, table.split-table td {{ border: 1px solid #ddd; padding: 4px 6px; vertical-align: top; }}
      table.split-table th {{ font-weight: 700; }}
    </style>
    <table class="split-table"><thead><tr>{header}</tr></thead><tbody>{''.join(body)}</tbody></table>{note}
    """))

ratio_sum = sum(SPLIT_RATIOS.values())
if any(value < 0 for value in SPLIT_RATIOS.values()) or abs(ratio_sum - 1.0) > 1e-9:
    raise ValueError(f"Split ratios must be non-negative and sum to 1.0; got {ratio_sum}")


## Load metadata about the entire dataset

In [4]:
# Read the extracted manifest and add fields needed for grouping.
with MANIFEST_CSV.open(newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    required_columns = {
        "sample_id",
        "source_h5",
        "source_index",
        "object_type",
        "is_gripping",
        "metadata_path",
        "rgb_path",
        "tactile_path",
    }
    missing_columns = required_columns.difference(reader.fieldnames or [])
    if missing_columns:
        raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")
    rows = list(reader)

for row in rows:
    row["source_index"] = int(row["source_index"])
    row["is_gripping"] = parse_bool(row["is_gripping"])
    row["episode_id"] = h5_number(row["source_h5"])

success_count = sum(row["is_gripping"] for row in rows)
manifest_summary = [{
    "samples": len(rows),
    "object_types": len({row["object_type"] for row in rows}),
    "files": len({row["episode_id"] for row in rows}),
    "success": success_count,
    "failure": len(rows) - success_count,
    "success_rate": round(success_count / len(rows), 4),
}]

display_table(manifest_summary, "Dataset summary")

samples,object_types,files,success,failure,success_rate
9296,105,38,5969,3327,0.6421


## Add a split_group field and assign data into train, eval and test datasets

In [5]:
# Aggregate samples into the selected split-group policy.
split_groups = defaultdict(lambda: {
    "object_type": "",
    "episode_id": "",
    "success": 0,
    "failure": 0,
    "source_h5_files": set(),
})

for row in rows:
    split_group = split_group_for(row)
    group = split_groups[split_group]
    group["object_type"] = row["object_type"]
    group["episode_id"] = "" if SPLIT_GROUP_STRATEGY == "object_type" else row["episode_id"]
    group["source_h5_files"].add(Path(row["source_h5"]).name)
    group["success"] += int(row["is_gripping"])
    group["failure"] += int(not row["is_gripping"])

split_group_table = []
for split_group, group in sorted(split_groups.items()):
    total_samples = group["success"] + group["failure"]
    split_group_table.append({
        "split_group": split_group,
        "object_type": group["object_type"],
        "episode_id": group["episode_id"],
        "is_gripping_success_count": group["success"],
        "is_gripping_failure_count": group["failure"],
        "total_samples": total_samples,
        "source_h5_files": ";".join(sorted(group["source_h5_files"])),
    })

sizes = sorted(row["total_samples"] for row in split_group_table)
display_table([{
    "strategy": SPLIT_GROUP_STRATEGY,
    "split_groups": len(split_group_table),
    "total_samples": sum(sizes),
    "largest_group": max(sizes),
    "median_group_size": sizes[len(sizes) // 2],
}], "Split group summary")
display_table(split_group_table, "Split groups")

# Shuffle whole groups, then assign groups until each target sample threshold is reached.
shuffled_groups = [dict(row) for row in split_group_table]
random.Random(SEED).shuffle(shuffled_groups)

total_samples = sum(row["total_samples"] for row in shuffled_groups)
train_cutoff = total_samples * SPLIT_RATIOS["train"]
eval_cutoff = train_cutoff + total_samples * SPLIT_RATIOS["eval"]

assigned_samples = 0
for row in shuffled_groups:
    if assigned_samples < train_cutoff:
        split = "train"
    elif assigned_samples < eval_cutoff:
        split = "eval"
    else:
        split = "test"
    row["split"] = split
    assigned_samples += row["total_samples"]

assigned_group_table = sorted(shuffled_groups, key=lambda row: row["split_group"])
assigned_summary = defaultdict(lambda: {"groups": 0, "samples": 0})
for row in assigned_group_table:
    assigned_summary[row["split"]]["groups"] += 1
    assigned_summary[row["split"]]["samples"] += row["total_samples"]

assignment_summary = []
for split, ratio in SPLIT_RATIOS.items():
    samples = assigned_summary[split]["samples"]
    assignment_summary.append({
        "split": split,
        "target_fraction": ratio,
        "target_samples": round(total_samples * ratio),
        "assigned_samples": samples,
        "assigned_fraction": round(samples / total_samples, 4),
        "assigned_groups": assigned_summary[split]["groups"],
    })

display_table(assignment_summary, "Target vs assigned split sizes")

strategy,split_groups,total_samples,largest_group,median_group_size
object_type,105,9296,772,42


split_group,object_type,episode_id,is_gripping_success_count,is_gripping_failure_count,total_samples,source_h5_files
3d_printed_black_cylinder_gear,3d_printed_black_cylinder_gear,,1,1,2,calandra_corl2017_015.h5
3d_printed_blue_connector,3d_printed_blue_connector,,54,18,72,calandra_corl2017_006.h5;calandra_corl2017_007.h5;calandra_corl2017_020.h5;calandra_corl2017_021.h5
3d_printed_blue_house,3d_printed_blue_house,,24,16,40,calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5
3d_printed_blue_vase,3d_printed_blue_vase,,58,0,58,calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5;calandra_corl2017_017.h5;calandra_corl2017_018.h5
3d_printed_white_ball,3d_printed_white_ball,,77,3,80,calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5
angry_bird,angry_bird,,14,26,40,calandra_corl2017_003.h5;calandra_corl2017_004.h5;calandra_corl2017_005.h5;calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5
aspirin,aspirin,,554,155,709,calandra_corl2017_003.h5;calandra_corl2017_004.h5;calandra_corl2017_005.h5;calandra_corl2017_008.h5;calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5;calandra_corl2017_018.h5;calandra_corl2017_019.h5;calandra_corl2017_020.h5;calandra_corl2017_022.h5;calandra_corl2017_023.h5;calandra_corl2017_026.h5;calandra_corl2017_027.h5;calandra_corl2017_035.h5;calandra_corl2017_036.h5;calandra_corl2017_037.h5
axe_body_spray,axe_body_spray,,0,6,6,calandra_corl2017_000.h5;calandra_corl2017_001.h5;calandra_corl2017_002.h5
baby_cup,baby_cup,,42,12,54,calandra_corl2017_008.h5;calandra_corl2017_009.h5;calandra_corl2017_012.h5;calandra_corl2017_013.h5;calandra_corl2017_014.h5;calandra_corl2017_015.h5;calandra_corl2017_016.h5
bag_pack,bag_pack,,25,13,38,calandra_corl2017_003.h5;calandra_corl2017_004.h5;calandra_corl2017_005.h5;calandra_corl2017_026.h5;calandra_corl2017_027.h5


split,target_fraction,target_samples,assigned_samples,assigned_fraction,assigned_groups
train,0.7,6507,6541,0.7036,76
eval,0.15,1394,1570,0.1689,16
test,0.15,1394,1185,0.1275,13


## Create Train, Eval and Test manifest files

In [6]:
# Attach assigned splits to sample rows and convert to the final training manifest schema.
visually_challenging_set = set(VISUALLY_CHALLENGING_OBJECT_TYPES)
split_lookup = {row["split_group"]: row["split"] for row in assigned_group_table}
final_manifest_rows = []

for row in rows:
    split_group = split_group_for(row)
    final_row = final_manifest_row(row, split_group, visually_challenging_set)
    final_row["split"] = split_lookup[split_group]
    final_manifest_rows.append(final_row)

row_summary = defaultdict(lambda: {"rows": 0, "success": 0, "split_groups": set()})
for row in final_manifest_rows:
    group = row_summary[row["split"]]
    group["rows"] += 1
    group["success"] += int(parse_bool(row["result"]))
    group["split_groups"].add(row["split_group"])

row_split_summary = []
for split in SPLIT_RATIOS:
    group = row_summary[split]
    row_split_summary.append({
        "split": split,
        "rows": group["rows"],
        "success": group["success"],
        "failure": group["rows"] - group["success"],
        "row_fraction": round(group["rows"] / len(final_manifest_rows), 4),
        "split_groups": len(group["split_groups"]),
    })

display_table(row_split_summary, "Data split")

split,rows,success,failure,row_fraction,split_groups
train,6541,4114,2427,0.7036,76
eval,1570,1001,569,0.1689,16
test,1185,854,331,0.1275,13


## Visually Challenging Objects

In [7]:
# Uncomment this helper if the object list should be refreshed from possibly-visually-difficult.csv.
# def load_visually_challenging_object_types(csv_path=Path("possibly-visually-difficult.csv")):
#     with csv_path.open(newline="") as csv_file:
#         reader = csv.DictReader(csv_file)
#         if "object_name" not in (reader.fieldnames or []):
#             raise ValueError("CSV is missing required column: object_name")
#         return [row["object_name"] for row in reader]
# VISUALLY_CHALLENGING_OBJECT_TYPES = load_visually_challenging_object_types()

# Filter to visually challenging objects and repeat the same split logic independently.
available_object_types = {row["object_type"] for row in rows}
missing_visually_challenging = sorted(visually_challenging_set - available_object_types)
if missing_visually_challenging:
    raise ValueError(f"Visually challenging object types not found in dataset: {missing_visually_challenging}")

visual_rows = [row for row in rows if row["object_type"] in visually_challenging_set]
visual_groups = defaultdict(lambda: {"object_type": "", "episode_id": "", "success": 0, "failure": 0})

for row in visual_rows:
    split_group = split_group_for(row)
    group = visual_groups[split_group]
    group["object_type"] = row["object_type"]
    group["episode_id"] = "" if SPLIT_GROUP_STRATEGY == "object_type" else row["episode_id"]
    group["success"] += int(row["is_gripping"])
    group["failure"] += int(not row["is_gripping"])

visual_group_table = []
for split_group, group in sorted(visual_groups.items()):
    total_samples = group["success"] + group["failure"]
    visual_group_table.append({
        "split_group": split_group,
        "object_type": group["object_type"],
        "episode_id": group["episode_id"],
        "is_gripping_success_count": group["success"],
        "is_gripping_failure_count": group["failure"],
        "total_samples": total_samples,
    })

visual_shuffled_groups = [dict(row) for row in visual_group_table]
random.Random(SEED).shuffle(visual_shuffled_groups)
visual_total_samples = sum(row["total_samples"] for row in visual_shuffled_groups)
visual_train_cutoff = visual_total_samples * SPLIT_RATIOS["train"]
visual_eval_cutoff = visual_train_cutoff + visual_total_samples * SPLIT_RATIOS["eval"]

assigned_samples = 0
for row in visual_shuffled_groups:
    if assigned_samples < visual_train_cutoff:
        split = "train"
    elif assigned_samples < visual_eval_cutoff:
        split = "eval"
    else:
        split = "test"
    row["split"] = split
    assigned_samples += row["total_samples"]

visual_assigned_group_table = sorted(visual_shuffled_groups, key=lambda row: row["split_group"])
visual_split_lookup = {row["split_group"]: row["split"] for row in visual_assigned_group_table}
visual_final_manifest_rows = []
for row in visual_rows:
    split_group = split_group_for(row)
    final_row = final_manifest_row(row, split_group, visually_challenging_set)
    final_row["split"] = visual_split_lookup[split_group]
    visual_final_manifest_rows.append(final_row)

visual_summary = defaultdict(lambda: {"rows": 0, "success": 0, "split_groups": set()})
for row in visual_final_manifest_rows:
    group = visual_summary[row["split"]]
    group["rows"] += 1
    group["success"] += int(parse_bool(row["result"]))
    group["split_groups"].add(row["split_group"])

visual_row_split_summary = []
for split in SPLIT_RATIOS:
    group = visual_summary[split]
    visual_row_split_summary.append({
        "split": split,
        "rows": group["rows"],
        "success": group["success"],
        "failure": group["rows"] - group["success"],
        "row_fraction": round(group["rows"] / len(visual_final_manifest_rows), 4),
        "split_groups": len(group["split_groups"]),
    })

display_table([{
    "visually_challenging_object_types": len(visually_challenging_set),
    "dataset_rows": len(visual_rows),
    "split_groups": len(visual_group_table),
}], "Visually challenging subset summary")
display_table(visual_row_split_summary, "Visually challenging row-level split summary")
display_table(visual_final_manifest_rows, "Visually challenging final manifest preview", limit=10)


visually_challenging_object_types,dataset_rows,split_groups
35,2353,35


split,rows,success,failure,row_fraction,split_groups
train,1796,1175,621,0.7633,20
eval,240,136,104,0.102,5
test,317,242,75,0.1347,10


episode_id,dataset_name,task_id,task_name,object_id,object_class,condition_class,clip_type,result,failure_mode,rgb_path,tactile_path,force_path,metadata_path,split_group,split
000,feeling_of_success,14,grasp_outcome,ponds_dry_skin_cream,visually_challenging,,before-during-after-images,True,,000_000014_ponds_dry_skin_cream/rgb,000_000014_ponds_dry_skin_cream/tactile,,000_000014_ponds_dry_skin_cream/metadata.json,ponds_dry_skin_cream,train
000,feeling_of_success,33,grasp_outcome,ponds_dry_skin_cream,visually_challenging,,before-during-after-images,False,unknown,000_000033_ponds_dry_skin_cream/rgb,000_000033_ponds_dry_skin_cream/tactile,,000_000033_ponds_dry_skin_cream/metadata.json,ponds_dry_skin_cream,train
000,feeling_of_success,41,grasp_outcome,tomato_paste_in_metal_can,visually_challenging,,before-during-after-images,True,,000_000041_tomato_paste_in_metal_can/rgb,000_000041_tomato_paste_in_metal_can/tactile,,000_000041_tomato_paste_in_metal_can/metadata.json,tomato_paste_in_metal_can,train
000,feeling_of_success,43,grasp_outcome,tomato_paste_in_metal_can,visually_challenging,,before-during-after-images,False,unknown,000_000043_tomato_paste_in_metal_can/rgb,000_000043_tomato_paste_in_metal_can/tactile,,000_000043_tomato_paste_in_metal_can/metadata.json,tomato_paste_in_metal_can,train
000,feeling_of_success,45,grasp_outcome,tomato_paste_in_metal_can,visually_challenging,,before-during-after-images,True,,000_000045_tomato_paste_in_metal_can/rgb,000_000045_tomato_paste_in_metal_can/tactile,,000_000045_tomato_paste_in_metal_can/metadata.json,tomato_paste_in_metal_can,train
000,feeling_of_success,60,grasp_outcome,candle_in_glass,visually_challenging,,before-during-after-images,False,unknown,000_000060_candle_in_glass/rgb,000_000060_candle_in_glass/tactile,,000_000060_candle_in_glass/metadata.json,candle_in_glass,test
000,feeling_of_success,76,grasp_outcome,ogx_shampoo,visually_challenging,,before-during-after-images,False,unknown,000_000076_ogx_shampoo/rgb,000_000076_ogx_shampoo/tactile,,000_000076_ogx_shampoo/metadata.json,ogx_shampoo,train
000,feeling_of_success,92,grasp_outcome,ponds_dry_skin_cream,visually_challenging,,before-during-after-images,False,unknown,000_000092_ponds_dry_skin_cream/rgb,000_000092_ponds_dry_skin_cream/tactile,,000_000092_ponds_dry_skin_cream/metadata.json,ponds_dry_skin_cream,train
000,feeling_of_success,94,grasp_outcome,tomato_paste_in_metal_can,visually_challenging,,before-during-after-images,False,unknown,000_000094_tomato_paste_in_metal_can/rgb,000_000094_tomato_paste_in_metal_can/tactile,,000_000094_tomato_paste_in_metal_can/metadata.json,tomato_paste_in_metal_can,train
000,feeling_of_success,108,grasp_outcome,ogx_shampoo,visually_challenging,,before-during-after-images,False,unknown,000_000108_ogx_shampoo/rgb,000_000108_ogx_shampoo/tactile,,000_000108_ogx_shampoo/metadata.json,ogx_shampoo,train


## CSV Export

In [8]:
# Write the final schema, omitting the temporary split column from each output file.
def write_final_split_csvs(source_rows, filenames_by_split):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for split_name, filename in filenames_by_split.items():
        output_path = OUTPUT_DIR / filename
        split_rows = [row for row in source_rows if row["split"] == split_name]
        with output_path.open("w", newline="") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=FINAL_MANIFEST_FIELDNAMES)
            writer.writeheader()
            for row in split_rows:
                writer.writerow({column: row[column] for column in FINAL_MANIFEST_FIELDNAMES})
        print(f"Wrote {len(split_rows):,} rows to {output_path}")

if WRITE_SPLIT_CSVS:
    write_final_split_csvs(final_manifest_rows, {
        "train": "train.csv",
        "eval": "eval.csv",
        "test": "test.csv",
    })
else:
    print("WRITE_SPLIT_CSVS is False; full dataset split CSV files not written.")

if WRITE_VISUALLY_CHALLENGING_SPLIT_CSVS:
    write_final_split_csvs(visual_final_manifest_rows, {
        "train": "visually_challenging_train.csv",
        "eval": "visually_challenging_eval.csv",
        "test": "visually_challenging_test.csv",
    })
else:
    print("WRITE_VISUALLY_CHALLENGING_SPLIT_CSVS is False; visually challenging split CSV files not written.")


Wrote 6,541 rows to ../feeling-of-success/train.csv
Wrote 1,570 rows to ../feeling-of-success/eval.csv
Wrote 1,185 rows to ../feeling-of-success/test.csv
Wrote 1,796 rows to ../feeling-of-success/visually_challenging_train.csv
Wrote 240 rows to ../feeling-of-success/visually_challenging_eval.csv
Wrote 317 rows to ../feeling-of-success/visually_challenging_test.csv
